# Notebook 01: Combining Data

## Overview
This notebook combines student dropout data from multiple sources (SDO1-SDO7) into a unified dataset for analysis.

### Data Sources:
- **SDO1**: Student characteristics, distance, previous education
- **SDO2**: SKC attendance and orientation events
- **SDO4**: DSA/NF degree information by college year
- **SDO5**: Student degree and dropout information with postal codes
- **SDO6**: Course results
- **SDO7**: Student wellbeing questionnaire

### Process Flow:
1. Load CSV files from raw data directory
2. Clean data (remove empty rows, unnamed columns)
3. Explore data quality and overlaps
4. Merge datasets on common keys
5. Export combined dataset for next notebooks

---

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from itertools import combinations
from collections import Counter, defaultdict

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

## 2. Load Raw Data

In [ ]:
# -------------------------------------------------------------------------
# Bulk CSV Loader (flat only, does not open subfolders of data) with Smart Encoding Handling
#
# What this script does:
# 1. Sets input (RAW_DIR) and output (OUT_DIR) directories.
# 2. Finds and loads all CSV files in RAW_DIR *only* (NO subfolders).
# 3. Handles tricky encodings (UTF-8 with BOM, fallback to Latin-1).
# 4. Keeps loading even if some files fail, and reports errors.
# 5. Stores loaded CSVs in a dictionary (tables) keyed by file name (no .csv).
# 6. Prints a summary: number of CSVs loaded and their names.
# -------------------------------------------------------------------------

# Define the path to the processed data based on the current user
current_user = os.getlogin()

# Define input and output directories (notebook is in ./notebooks; data is ../data)
DATA_DIR     = Path(rf"/home/{current_user}/local-share")
RAW_DIR      = DATA_DIR / "raw"
OUT_DIR      = DATA_DIR / "processed"

# Import processed data from csv

PROC_DIR = Path(rf"/home/{current_user}/local-share")

# --- Helpers ---
def read_csv_smart(path: Path, **kwargs) -> pd.DataFrame:
    """Read CSV with sensible defaults, trying UTF-8 first and Latin-1 as fallback."""
    try:
        return pd.read_csv(path, encoding="utf-8-sig", **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", **kwargs)

def load_csvs_flat(base: Path, **kwargs) -> dict[str, pd.DataFrame]:
    """
    Load CSVs located directly in `base` (no subfolders) into a dict keyed by filename (without .csv).
    Prints how many tables were loaded and how many errors occurred.
    """
    tables, errors = {}, []
    for p in sorted(base.glob("*.csv")):  # <-- flat search only
        key = p.stem  # filename without extension
        try:
            tables[key] = read_csv_smart(p, **kwargs)
        except Exception as e:
            errors.append((str(p), str(e)))
    print(f"Loaded {len(tables)} tables; errors: {len(errors)}")
    if errors:
        for name, err in errors[:10]:
            print(" -", name, "->", err)
    return tables

# --- Load CSV files in RAW_DIR (flat only) ---
tables = load_csvs_flat(
    RAW_DIR,
    sep=None,        # auto-detect delimiter
    engine="python", # required for sep=None sniffing
)

# Print summary in the requested format
print(f"\nNumber of csv files found: {len(tables)}")
print("List of csv files:")
for i, name in enumerate(tables.keys(), start=1):
    print(f"{i}. {name}")


### 2.1 Data Summary

In [ ]:
# show summary of data files
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_seq_items", None)   # don't truncate lists

summary = (
    pd.DataFrame(
        [{
            "key": k,
            "rows": len(df),
            "cols": df.shape[1],
            "columns": list(map(str, df.columns))  # full list, no manual truncation
        } for k, df in tables.items()]
    ).sort_values("key")
)

summary


### 2.2 Create Named DataFrames

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Create globally accessible pandas DataFrames with clean, consistent
# names from a `tables` source dict. For each (clean_name -> source_key) pair,
# this script fetches the DataFrame from `tables`, assigns it into `globals()`
# under `clean_name`, and logs what was created. If a source key is missing,
# it logs a warning.
# -----------------------------------------------------------------------------

# Step 1: Map clean DataFrame names -> source keys in `tables`
custom_names = {
    "sdo1_distance": "sdo1-helperdata-Euclidean_distance_postcode_to_HU",
    "sdo1_characteristics": "sdo1-student_characteristics",
    "sdo1_previous": "sdo1-student_previous_education",
    "sdo2_skc": "sdo2-skc",
    "sdo2_orientation": "sdo2-student_orientation",
    "sdo4_dsa": "sdo4-DSA_degree_collegeyear_2018-2023",
    "sdo5_degree": "sdo5-student_degree_drop-out_postalcode",
    "sdo6_results": "sdo6-course_results",
    "sdo7_questionaire": "sdo78-student_wellbeing",
    "sdo4_nf": "sdo4-NF_degree_collegeyear"
}

# Step 2: Create DataFrames with clean names (keep everything)
for clean_name, source_key in custom_names.items():
    df = tables.get(source_key)
    if df is None:
        print(f"[warning] '{source_key}' not found in tables.")
        continue

    globals()[clean_name] = df
    print(f"Created DataFrame: {clean_name} from '{source_key}' with {len(df)} rows")


## 3. Data Cleaning

### 3.1 Remove Fully Empty Rows

------------------------- For each dataframe, drop rows that are entirely empty across all columns ----------------------------------

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: For each loaded DataFrame, convert empty/whitespace-only strings to NaN
# (object/string columns only) and DROP rows that are entirely empty (all-NaN).
# Prints a before/after summary per dataset. Safe to run multiple times.
# -----------------------------------------------------------------------------

df_names = [
    "sdo1_distance",
    "sdo1_characteristics",
    "sdo1_previous",
    "sdo2_skc",
    "sdo2_orientation",
    "sdo4_dsa",
    "sdo5_degree",
    "sdo6_results",
    "sdo7_questionaire",
    "sdo4_nf"
]

def _drop_fully_empty_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    # Treat pure-whitespace as empty only for textual columns
    obj_cols = df.select_dtypes(include=["object", "string"]).columns
    if len(obj_cols):
        df = df.copy()
        df[obj_cols] = df[obj_cols].replace(r"^\s*$", np.nan, regex=True)

    empty_mask = df.isna().all(axis=1)
    n_empty = int(empty_mask.sum())
    return df.loc[~empty_mask].copy() if n_empty else df, n_empty

for name in df_names:
    df = globals().get(name)
    if not isinstance(df, pd.DataFrame):
        print(f"[warning] '{name}' not found or not a DataFrame; skipping.")
        continue

    before = len(df)
    cleaned, dropped = _drop_fully_empty_rows(df)
    globals()[name] = cleaned
    after = len(cleaned)

    print(f"{name}: dropped {dropped} fully-empty rows (rows {before} → {after})")

# Assert no fully empty rows remain in any dataset
for name in df_names:
    df = globals().get(name)
    if isinstance(df, pd.DataFrame):
        assert not df.isna().all(axis=1).any(), f"{name} still has fully empty rows."


Anticipation: 

Fully empty rows usually come from source files (especially Excel) where the sheet’s UsedRange includes formatted-but-empty rows or trailing delimiters, 
so the importer reads them as blank records. They can also be created downstream by outer joins/concats with mismatched keys or 
by converting whitespace/empty strings to NaN across all columns, leaving rows with nothing but missing values.

### 3.2 Remove Unnamed Columns

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Drop any "unnamed" columns (e.g., "Unnamed: 0", empty-string headers)
# from all loaded DataFrames. Safe to run multiple times; prints what was removed.
# -----------------------------------------------------------------------------

df_names = [
    "sdo1_distance",
    "sdo1_characteristics",
    "sdo1_previous",
    "sdo2_skc",
    "sdo2_orientation",
    "sdo4_dsa",
    "sdo5_degree",
    "sdo6_results",
    "sdo7_questionaire",
    "sdo4_nf"
]

def _unnamed_cols(df: pd.DataFrame):
    cols = []
    for c in df.columns:
        s = str(c)
        if s.strip() == "" or s.lower().startswith("unnamed"):
            cols.append(c)
    return cols

for name in df_names:
    df = globals().get(name)
    if not isinstance(df, pd.DataFrame):
        print(f"[warning] '{name}' not found or not a DataFrame; skipping.")
        continue

    to_drop = _unnamed_cols(df)
    if to_drop:
        globals()[name] = df.drop(columns=to_drop)
        print(f"{name}: dropped {len(to_drop)} unnamed columns -> {to_drop}")
    else:
        print(f"{name}: no unnamed columns found.")


## 4. Data Quality Analysis

### 4.1 Check Missing Values in SDO2 Data

SDO2 data = `sdo2_skc` and `sdo2_orientation`

In [ ]:
# Confirm NaN count per column in sdo2_skc
skc_nan_sums = sdo2_skc.isna().sum().astype(int).sort_values(ascending=False).head(20)
print(skc_nan_sums)
print("sdo2_skc shape is:", sdo2_skc.shape)

In [ ]:
# Confirm NaN count per column in sdo2_orientation
orientation_nan_sums = sdo2_orientation.isna().sum().astype(int).sort_values(ascending=False).head(20)
print(orientation_nan_sums)
print("sdo2_orientation shape is:", sdo2_orientation.shape)


In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Check overlap between students in sdo2_skc and sdo2_orientation.
# Shows counts and percentages of unique, shared, and total SINH_IDs.
# -----------------------------------------------------------------------------

# Get sets of student IDs
ids_skc = set(sdo2_skc['SINH_ID'].dropna())
ids_ori = set(sdo2_orientation['SINH_ID'].dropna())

# Compute intersections and differences
shared_ids = ids_skc & ids_ori
only_skc   = ids_skc - ids_ori
only_ori   = ids_ori - ids_skc

# Print summary statistics
print(f"Total in sdo2_skc: {len(ids_skc):,}")
print(f"Total in sdo2_orientation: {len(ids_ori):,}")
print(f"Shared students: {len(shared_ids):,}")
print(f"Only in sdo2_skc: {len(only_skc):,}")
print(f"Only in sdo2_orientation: {len(only_ori):,}")

# Optional: show overlap percentage
overlap_pct = len(shared_ids) / len(ids_skc | ids_ori) * 100
print(f"\nOverlap percentage: {overlap_pct:.2f}%")

### 4.2 Check Student ID Overlaps

Anticipation of NaN: 

After merging, the high (~80%) NaN values in sdo2 columns are likely correct.

This is because even before merge, we can see there are a lot of empty values and 

hardly a good overlap between the sdo2 data in the first place, only 8,583 students overlap. 

Thus, the poor overlapp was dragged into the merging which could explain the (~80%) NaN values.

This happens because the merge keeps all 47,950 students, but 

many have no matching records in sdo2_skc or sdo2_orientation or both.

Therefore, the problem could come from data collection.

Similary,  a total of 3476 NaN in gender could be from data collection.

This a lot but relatively (compared to the total size of the data) its not that much, 

It wont have an effect on our analysis because we have enough data.

if its going to give more headache to resolve, we can exclude it , we are still going 

to have more than 40k observations. 

## 5. Data Exploration and Validation

In [ ]:
sdo5_degree.columns

In [ ]:
sdo2_orientation.columns

In [ ]:
# -----------------------------------------------------------------------------
# PURPOSE
# -----------------------------------------------------------------------------
# Merge `sdo2_orientation` with `sdo5_degree` on SINH_ID to bring in COLLEGEJAAR,
# then compute what percentage of degree students have ANY orientation record
# with First_Event_Date in year ≥ 2020.
#
# Notes:
# - Counts are per student (unique SINH_ID), not per event row.
# - Left-join from degrees → denominator is all degree students.
# - Also prints an optional breakdown by COLLEGEJAAR.
# -----------------------------------------------------------------------------

# Defensive copies
ori = sdo2_orientation.copy()
deg = sdo5_degree.copy()

# Ensure expected columns exist (based on renaming above)
required_ori = {'SINH_ID','First_Event_Date'}
required_deg = {'SINH_ID','COLLEGEJAAR'}
missing = (required_ori - set(ori.columns)) | (required_deg - set(deg.columns))
if missing:
    raise KeyError(f"Missing required columns: {missing}")

# Parse orientation date and flag year ≥ 2020 per student
ori['First_Event_Date'] = pd.to_datetime(ori['First_Event_Date'], errors='coerce')
ori['year'] = ori['First_Event_Date'].dt.year
ori_flag = (
    ori.assign(has_orient_2020plus = ori['year'] >= 2020)
       .groupby('SINH_ID', as_index=False)['has_orient_2020plus']
       .max()  # True if ANY record in ≥2020
)

# Use one row per student from degree table for clean denominator
deg_unique = deg[['SINH_ID','COLLEGEJAAR']].drop_duplicates('SINH_ID')

# Merge and compute percentage
merged = deg_unique.merge(ori_flag, on='SINH_ID', how='left')
merged['has_orient_2020plus'] = merged['has_orient_2020plus'].fillna(False)

total_students = merged['SINH_ID'].nunique()
num_with_2020plus = int(merged['has_orient_2020plus'].sum())
pct_with_2020plus = (num_with_2020plus / total_students * 100) if total_students else 0.0

print(f"Students with orientation data in year ≥ 2020: {num_with_2020plus:,} of {total_students:,} "
      f"({pct_with_2020plus:.2f}%).")

# Optional: breakdown by college year
by_cy = (
    merged.groupby('COLLEGEJAAR', dropna=False)['has_orient_2020plus']
          .mean()
          .mul(100)
          .sort_index()
          .round(2)
)
print("\nPercentage by COLLEGEJAAR (year ≥ 2020):")
print(by_cy)


### 5.1 Analyze Column Similarities Across Datasets

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Normalize SINH_ID naming across all datasets and verify primary keys
# per-DataFrame (no mixing). For frames with SINH_ID (in any case/spacing form),
# rename to 'SINH_ID' and test PK (nulls + duplicates). For frames that lack
# SINH_ID (e.g., sdo1_distance, sdo4_dsa), search for a composite key (pairs,
# optionally triples) and report findings with diagnostics.
# -----------------------------------------------------------------------------

# --- 1) Rebuild registry safely (no mixing) ---
clean_names = list(custom_names.keys())
df_registry = {}
for name in clean_names:
    df = globals().get(name, None)
    if isinstance(df, pd.DataFrame):
        df_registry[name] = df
    else:
        print(f"[warning] DataFrame '{name}' not found. Skipping.")

# --- 2) Helpers ---
def _canonicalize(s: str) -> str:
    """Lowercase and remove non-alphanumerics for fuzzy header matching."""
    return re.sub(r'[^a-z0-9]', '', s.lower()) if isinstance(s, str) else s

def find_sinh_id_column(cols) -> str | None:
    """
    Detect a column that semantically matches SINH_ID even if the header's case,
    spacing or punctuation varies, e.g., 'sinh_id', 'Sinh Id', 'SINH-ID'.
    """
    want = "sinhid"
    for c in cols:
        if _canonicalize(c) == want:
            return c
    return None

def robust_min_max(s: pd.Series):
    if is_numeric_dtype(s):
        c = pd.to_numeric(s, errors="coerce")
        return c.min(), c.max()
    if is_datetime64_any_dtype(s):
        c = pd.to_datetime(s, errors="coerce")
        return c.min(), c.max()
    return None, None

def search_composite_key(df: pd.DataFrame, try_triples: bool = False, max_combos: int = 5000):
    """
    Attempt to find a valid composite key (pairs, optionally triples).
    Returns (candidate_cols: tuple|None, null_rows, dup_rows).
    """
    tested = 0

    # Pairs first
    for cols in combinations(df.columns, 2):
        if tested >= max_combos:
            break
        tested += 1
        sub = df[list(cols)]
        null_rows = int(sub.isna().any(axis=1).sum())
        dup_rows = int(df.duplicated(subset=list(cols)).sum())
        if null_rows == 0 and dup_rows == 0:
            return cols, null_rows, dup_rows

    if not try_triples:
        return None, None, None

    # Triples (can be heavy)
    for cols in combinations(df.columns, 3):
        if tested >= max_combos:
            break
        tested += 1
        sub = df[list(cols)]
        null_rows = int(sub.isna().any(axis=1).sum())
        dup_rows = int(df.duplicated(subset=list(cols)).sum())
        if null_rows == 0 and dup_rows == 0:
            return cols, null_rows, dup_rows

    return None, None, None

# --- 3) Normalize SINH_ID headers in-place, then test PKs ---
rename_log = []
pk_rows = []
dup_samples = {}  # dataset -> small df of duplicated key values (if any)

for ds_name, df in df_registry.items():
    # 3a) Normalize header to 'SINH_ID' if a fuzzy match exists
    match = find_sinh_id_column(df.columns)
    if match and match != "SINH_ID":
        df.rename(columns={match: "SINH_ID"}, inplace=True)
        rename_log.append((ds_name, match, "SINH_ID"))

    # 3b) Test PKs
    if "SINH_ID" in df.columns:
        nulls = int(df["SINH_ID"].isna().sum())
        dup_rows = int(df.duplicated(subset=["SINH_ID"]).sum())
        is_valid = (nulls == 0) and (dup_rows == 0)

        pk_rows.append({
            "dataset": ds_name,
            "has_SINH_ID": True,
            "pk_type": "single",
            "pk_columns": ("SINH_ID",),
            "null_rows": nulls,
            "dup_rows": dup_rows,
            "is_valid_pk": bool(is_valid),
        })

        # If invalid, capture a small sample of duplicates for inspection
        if not is_valid and dup_rows > 0:
            dupe_vals = (
                df["SINH_ID"]
                .value_counts(dropna=False)
                .loc[lambda s: s > 1]
                .head(10)
                .rename("count")
                .reset_index()
                .rename(columns={"index": "SINH_ID"})
            )
            dup_samples[ds_name] = dupe_vals

    else:
        # No SINH_ID present (e.g., sdo1_distance, sdo4_dsa) — search for composite
        cand_cols, nulls, dup_rows = search_composite_key(df, try_triples=False, max_combos=5000)
        is_valid = cand_cols is not None

        pk_rows.append({
            "dataset": ds_name,
            "has_SINH_ID": False,
            "pk_type": ("pair" if cand_cols and len(cand_cols) == 2 else
                        "triple" if cand_cols and len(cand_cols) == 3 else None),
            "pk_columns": cand_cols,
            "null_rows": int(nulls) if nulls is not None else None,
            "dup_rows": int(dup_rows) if dup_rows is not None else None,
            "is_valid_pk": bool(is_valid),
        })

# --- 4) Assemble reports ---
pk_report = (
    pd.DataFrame(pk_rows)
    .sort_values(["has_SINH_ID", "dataset"], ascending=[False, True])
    .reset_index(drop=True)
)

# Optional: a quick, filtered view of what you likely want to confirm
expected_pk_view = pk_report.loc[pk_report["has_SINH_ID"], ["dataset", "pk_columns", "null_rows", "dup_rows", "is_valid_pk"]]

# Emit logs so you can see what got renamed and any dup samples available
if rename_log:
    print("Renamed headers to normalize SINH_ID:")
    for ds, old, new in rename_log:
        print(f"  - {ds}: '{old}' -> '{new}'")

print("\nPrimary key validation summary (single SINH_ID where present; composite search otherwise):")
display(pk_report)

# If you want to quickly inspect duplicate SINH_IDs:
for ds_name, dup_df in dup_samples.items():
    print(f"\nTop duplicated SINH_IDs in {ds_name}:")
    display(dup_df)

# Example: enforce index for frames with valid single-column PK
for ds_name, df in df_registry.items():
    if "SINH_ID" in df.columns:
        if df["SINH_ID"].isna().sum() == 0 and df.duplicated(subset=["SINH_ID"]).sum() == 0:
            try:
                df.set_index("SINH_ID", inplace=True, drop=False)  # keep column + index
                print(f"Index set to SINH_ID for {ds_name}")
            except Exception as e:
                print(f"[warning] Could not set index for {ds_name}: {e}")


### 5.2 Identify Potential Join Keys

Examine which columns could serve as primary or foreign keys for merging datasets.

## 6. Data Merging Strategy

### 6.1 Define Base Dataset and Keys

### 6.2 Examine Base Dataset Structure

### 6.3 Check for Duplicate Keys

In [ ]:
sdo1_previous

In [ ]:
sdo1_previous.columns

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Make SINH_ID the primary key in sdo1_previous by keeping only the row
# with the LATEST 'Final Exam Date' per SINH_ID. Rows with missing SINH_ID are dropped.
# Dates are parsed day-first (e.g., 28-06-2017). NaT dates are never selected when a
# real date exists for that SINH_ID.
# -----------------------------------------------------------------------------

df = sdo1_previous.copy()

# Parse dates (European format) and standardize SINH_ID
df["Final Exam Date"] = pd.to_datetime(df["Final Exam Date"], dayfirst=True, errors="coerce")
df["SINH_ID"] = pd.to_numeric(df["SINH_ID"], errors="coerce").round().astype("Int64")

# Drop rows without SINH_ID (cannot be a PK)
df = df.dropna(subset=["SINH_ID"]).copy()

# Sort so real dates come after NaT; keeping 'last' selects the latest real date
df = df.sort_values(["SINH_ID", "Final Exam Date"], ascending=[True, True], na_position="first")

# Keep only the latest row per SINH_ID
sdo1_previous = df.drop_duplicates(subset=["SINH_ID"], keep="last").copy()

# Sanity checks
assert sdo1_previous["SINH_ID"].isna().sum() == 0, "SINH_ID still has NaNs after filtering."
assert sdo1_previous.duplicated(subset=["SINH_ID"]).sum() == 0, "SINH_ID not unique after deduping."

print("sdo1_previous now unique on SINH_ID with latest Final Exam Date per student.")
print("Rows:", len(sdo1_previous))


In [ ]:
sdo1_previous

### 6.4 Analyze Duplicate Records

In [ ]:
sdo2_orientation

In [ ]:
# For the sdo2_orientation data frame, check how many NaN are there in the SINH_ID column

nan_count = int(sdo2_orientation["SINH_ID"].isna().sum())
print(nan_count)

In [ ]:
# Check the count of rows with a NaN in at least two columns in the data frame.

count_at_least_two_nan = int((sdo2_orientation.isna().sum(axis=1) >= 2).sum())
print(count_at_least_two_nan)


In [ ]:

# Check the count of rows with a NaN in at least three columns in the data frame.

count_at_least_two_nan = int((sdo2_orientation.isna().sum(axis=1) >= 3).sum())
print(count_at_least_two_nan)


In [ ]:
# NaN count per column in sdo2_orientation
nan_sums = sdo2_orientation.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

In [ ]:
print(sdo2_orientation["Number_of_Events_Attended"].value_counts(dropna=False))

In [ ]:
print(sdo2_orientation["Number_of_Event_Types"].value_counts(dropna=False))

In [ ]:
# NaN count per column in sdo2_orientation
nan_sums = sdo2_orientation.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

In [ ]:
# Convert SINH_ID in sdo2_orientation to integer, coercing errors to NaN
sdo2_orientation["SINH_ID"] = pd.to_numeric(sdo2_orientation["SINH_ID"], errors="coerce").round().astype("Int64")

# Drop NA SINH_ID rows
sdo2_orientation = sdo2_orientation.dropna(subset=["SINH_ID"]).copy()

### 6.5 Validate Primary Keys Across Datasets

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Standardize column names and add the DATAFRAME NAME as a PREFIX
# (provenance) in an *idempotent* way. Re-running this cell will not double-prefix.
#
# Steps per DataFrame:
#   1) Convert column names to underscored form (e.g., "Final Exam Date" -> "Final_Exam_Date").
#   2) For NON-key columns, ensure they are prefixed with "<dataset>_" (e.g., "sdo1_previous_Final_Exam_Date").
#   3) Do NOT change preserved join keys (default: 'SINH_ID').
#   4) Avoid double changes:
#        • If a column already has the desired "<dataset>_" prefix, leave it.
#        • If it has a trailing "_<dataset>" suffix from earlier steps, strip it before adding the prefix.
#   5) Ensure final names are unique (add __2, __3 if collisions occur).
# -----------------------------------------------------------------------------


df_names = [
    "sdo1_distance",
    "sdo1_characteristics",
    "sdo1_previous",
    "sdo2_skc",
    "sdo2_orientation",
    "sdo4_dsa",
    "sdo5_degree",
    "sdo6_results",
    "sdo7_questionaire",
    "sdo4_nf"
]

# Join keys to KEEP as-is (no prefix)
preserve_keys = {"SINH_ID"}  # add more if needed, e.g., {"SINH_ID", "POSTCODE"}

def underscore_words(name: str) -> str:
    s = str(name)
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s)        # camelCase -> camel_Case
    s = re.sub(r"([A-Za-z])([0-9])", r"\1_\2", s)        # A1 -> A_1
    s = re.sub(r"([0-9])([A-Za-z])", r"\1_\2", s)        # 1A -> 1_A
    s = re.sub(r"[^0-9A-Za-z]+", "_", s)                 # spaces/punct -> _
    s = re.sub(r"_+", "_", s).strip("_")                 # collapse/trim _
    return s

# Normalize preserved keys for comparisons after underscoring
preserve_keys_norm = {underscore_words(k) for k in preserve_keys}

for name in df_names:
    df = globals().get(name)
    if not isinstance(df, pd.DataFrame):
        print(f"[warning] {name} not found or not a DataFrame; skipping.")
        continue

    prefix = f"{name}_"
    suffix = f"_{name}"

    current_cols = list(df.columns)

    # If ALL non-key columns already start with "<dataset>_", we consider it done (idempotent).
    non_key_cols = [c for c in current_cols if underscore_words(c) not in preserve_keys_norm]
    if non_key_cols and all(str(c).startswith(prefix) for c in non_key_cols):
        print(f"{name}: already prefixed; no changes made.")
        continue

    col_map = {}
    for orig in current_cols:
        base = underscore_words(orig)

        # If previous step added a suffix (e.g., base_<dataset>), strip it first
        if base.endswith(suffix):
            base = base[: -len(suffix)]

        # If the desired prefix is already present, leave it
        if base.startswith(prefix):
            target = base

        # Preserve join keys exactly (no prefix)
        elif base in preserve_keys_norm:
            target = base
            # If someone previously prefixed/suffixed a key, leave as-is to avoid breaking joins
            if str(orig).startswith(prefix) or str(orig).endswith(suffix):
                # no rename for keys that already carry dataset tags
                continue

        else:
            # Add the dataset prefix
            target = f"{prefix}{base}"

        if orig != target:
            col_map[orig] = target

    if not col_map:
        print(f"{name}: no columns needed renaming; skipped.")
        continue

    # Ensure uniqueness of targets (disambiguate collisions if any)
    proposed = list(col_map.values())
    counts = Counter(proposed)
    if any(cnt > 1 for cnt in counts.values()):
        seen = defaultdict(int)
        for k, v in list(col_map.items()):
            seen[v] += 1
            if counts[v] > 1:
                if seen[v] == 1:
                    continue  # first keeps base
                col_map[k] = f"{v}__{seen[v]}"

    globals()[name] = df.rename(columns=col_map)
    print(f"{name}: renamed {len(col_map)} column(s); keys preserved: {sorted(preserve_keys)}")


In [ ]:
sdo2_orientation

In [ ]:
sdo1_distance

In [ ]:
# check number of rows and columns in each dataset 
df_names = [
    "sdo1_distance",
    "sdo1_characteristics",
    "sdo1_previous",
    "sdo2_skc",
    "sdo2_orientation",
    "sdo4_dsa",
    "sdo5_degree",
    "sdo6_results",
    "sdo7_questionaire",
    "sdo4_nf"
]

summary = []
for name in df_names:
    df = globals().get(name)
    if isinstance(df, pd.DataFrame):
        r, c = df.shape
        summary.append({"dataset": name, "rows": r, "cols": c})
    else:
        print(f"[warning] {name} not found or not a DataFrame")

# Optional compact table (comment out if you only want the prints)
shape_report = pd.DataFrame(summary).sort_values("dataset").reset_index(drop=True)
shape_report


## 7. Merging Datasets

### 7.1 Merge SDO1 Data (Characteristics, Distance, Previous Education)

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Normalize SINH_ID naming across all datasets and verify primary keys
# per-DataFrame (no mixing). For frames with SINH_ID (in any case/spacing form),
# rename to 'SINH_ID' and test PK (nulls + duplicates). For frames that lack
# SINH_ID (e.g., sdo1_distance, sdo4_dsa), search for a composite key (pairs,
# optionally triples) and report findings with diagnostics.
# -----------------------------------------------------------------------------

# --- 1) Rebuild registry safely (no mixing) ---
clean_names = list(custom_names.keys())
df_registry = {}
for name in clean_names:
    df = globals().get(name, None)
    if isinstance(df, pd.DataFrame):
        df_registry[name] = df
    else:
        print(f"[warning] DataFrame '{name}' not found. Skipping.")

# --- 2) Helpers ---
def _canonicalize(s: str) -> str:
    """Lowercase and remove non-alphanumerics for fuzzy header matching."""
    return re.sub(r'[^a-z0-9]', '', s.lower()) if isinstance(s, str) else s

def find_sinh_id_column(cols) -> str | None:
    """
    Detect a column that semantically matches SINH_ID even if the header's case,
    spacing or punctuation varies, e.g., 'sinh_id', 'Sinh Id', 'SINH-ID'.
    """
    want = "sinhid"
    for c in cols:
        if _canonicalize(c) == want:
            return c
    return None

def robust_min_max(s: pd.Series):
    if is_numeric_dtype(s):
        c = pd.to_numeric(s, errors="coerce")
        return c.min(), c.max()
    if is_datetime64_any_dtype(s):
        c = pd.to_datetime(s, errors="coerce")
        return c.min(), c.max()
    return None, None


def search_composite_key(df: pd.DataFrame, try_triples: bool = False, max_combos: int = 5000):
    """
    Attempt to find a valid composite key (pairs, optionally triples).
    Returns (candidate_cols: tuple|None, null_rows, dup_rows).
    """
    tested = 0

    # Pairs first
    for cols in combinations(df.columns, 2):
        if tested >= max_combos:
            break
        tested += 1
        sub = df[list(cols)]
        null_rows = int(sub.isna().any(axis=1).sum())
        dup_rows = int(df.duplicated(subset=list(cols)).sum())
        if null_rows == 0 and dup_rows == 0:
            return cols, null_rows, dup_rows

    if not try_triples:
        return None, None, None

    # Triples (can be heavy)
    for cols in combinations(df.columns, 3):
        if tested >= max_combos:
            break
        tested += 1
        sub = df[list(cols)]
        null_rows = int(sub.isna().any(axis=1).sum())
        dup_rows = int(df.duplicated(subset=list(cols)).sum())
        if null_rows == 0 and dup_rows == 0:
            return cols, null_rows, dup_rows

    return None, None, None

# --- 3) Normalize SINH_ID headers in-place, then test PKs ---
rename_log = []
pk_rows = []
dup_samples = {}  # dataset -> small df of duplicated key values (if any)

# Add a 'rows' column (rowcount per DataFrame) and show it right after 'dataset'

# ... keep everything above unchanged ...

for ds_name, df in df_registry.items():
    # 3a) Normalize header to 'SINH_ID' if a fuzzy match exists
    match = find_sinh_id_column(df.columns)
    if match and match != "SINH_ID":
        df.rename(columns={match: "SINH_ID"}, inplace=True)
        rename_log.append((ds_name, match, "SINH_ID"))

    # 3b) Test PKs
    n_rows = int(len(df))  # <-- rowcount

    if "SINH_ID" in df.columns:
        nulls = int(df["SINH_ID"].isna().sum())
        dup_rows = int(df.duplicated(subset=["SINH_ID"]).sum())
        is_valid = (nulls == 0) and (dup_rows == 0)

        pk_rows.append({
            "dataset": ds_name,
            "rows": n_rows,                 # <-- add here
            "has_SINH_ID": True,
            "pk_type": "single",
            "pk_columns": ("SINH_ID",),
            "null_rows": nulls,
            "dup_rows": dup_rows,
            "is_valid_pk": bool(is_valid),
        })

        if not is_valid and dup_rows > 0:
            dupe_vals = (
                df["SINH_ID"]
                .value_counts(dropna=False)
                .loc[lambda s: s > 1]
                .head(10)
                .rename("count")
                .reset_index()
                .rename(columns={"index": "SINH_ID"})
            )
            dup_samples[ds_name] = dupe_vals

    else:
        cand_cols, nulls, dup_rows = search_composite_key(df, try_triples=False, max_combos=5000)
        is_valid = cand_cols is not None

        pk_rows.append({
            "dataset": ds_name,
            "rows": n_rows,                 # <-- add here
            "has_SINH_ID": False,
            "pk_type": ("pair" if cand_cols and len(cand_cols) == 2 else
                        "triple" if cand_cols and len(cand_cols) == 3 else None),
            "pk_columns": cand_cols,
            "null_rows": int(nulls) if nulls is not None else None,
            "dup_rows": int(dup_rows) if dup_rows is not None else None,
            "is_valid_pk": bool(is_valid),
        })

# --- 4) Assemble reports (show 'rows' right after 'dataset') ---
pk_report = (
    pd.DataFrame(pk_rows)
      .sort_values(["has_SINH_ID", "dataset"], ascending=[False, True])
      .reset_index(drop=True)
)

# Reorder columns for display: dataset, rows, then the rest
cols_order = ["dataset", "rows", "has_SINH_ID", "pk_type", "pk_columns", "null_rows", "dup_rows", "is_valid_pk"]
pk_report = pk_report.loc[:, cols_order]

print("\nPrimary key validation summary (single SINH_ID where present; composite search otherwise):")
display(pk_report)


### 7.2 Validate Merge Results

Check if the merge preserved all expected records and keys.

In [ ]:
# Remove sdo7_questionaire DataFrame from the global namespace
globals().pop("sdo7_questionaire", None)  # safe even if it doesn't exist

# Optional: free memory aggressively
import gc; gc.collect()

### 7.3 Merge Additional Datasets

Merge remaining datasets (orientation, results, SKC, etc.) with proper validation.

#### 7.3.1 Merge Orientation Data

In [ ]:
# sdo1_previous has 361,737 observations? 
sdo1_previous.SINH_ID.nunique()

In [ ]:
sdo2_orientation.columns

#### 7.3.2 First merge

In [ ]:
# ---------------------------------------------------------------------------
# Build first_merge strictly from:
#   sdo5_degree  (BASE, keep all columns)
#   + sdo1_characteristics
#   + sdo1_previous
#   + sdo2_skc
#   + sdo2_orientation
#   + sdo6_results
# All joins are LEFT on SINH_ID with NO suffixes allowed.
# Adds presence flags at the end.
# ---------------------------------------------------------------------------


def ensure_plain_column_key(df, key="SINH_ID"):
    df = df.copy()
    if isinstance(df.index, pd.MultiIndex):
        if key in (df.index.names or []):
            df = df.reset_index(drop=(key in df.columns))
    elif getattr(df.index, "name", None) == key:
        df = df.reset_index(drop=(key in df.columns))
    return df.rename_axis(None)

def as_int64(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")

def assert_no_overlap(left: pd.DataFrame, right: pd.DataFrame, key: str = "SINH_ID"):
    overlap = (set(left.columns) & set(right.columns)) - {key}
    if overlap:
        raise ValueError(
            f"Refusing to merge because these columns would collide without suffixes: {sorted(overlap)}.\n"
            f"Rename/drop before merging."
        )

def left_merge_no_suffix(left: pd.DataFrame, right: pd.DataFrame, on: str = "SINH_ID"):
    assert_no_overlap(left, right, on)
    before = len(left)
    out = left.merge(right, on=on, how="left", validate="one_to_one")
    assert len(out) == before, f"Row count changed on merge with key '{on}'"
    return out

# ----------------- Base -----------------
first_merge = ensure_plain_column_key(sdo5_degree, "SINH_ID").copy()
first_merge["SINH_ID"] = as_int64(first_merge["SINH_ID"])
base_n = len(first_merge)

# ----------------- Prepare right-hand tables (SINH_ID only) -----------------
char     = ensure_plain_column_key(sdo1_characteristics, "SINH_ID").copy()
prev     = ensure_plain_column_key(sdo1_previous,       "SINH_ID").copy()
skc      = ensure_plain_column_key(sdo2_skc,            "SINH_ID").copy()
orient   = ensure_plain_column_key(sdo2_orientation,    "SINH_ID").copy()
results  = ensure_plain_column_key(sdo6_results,        "SINH_ID").copy()

for df_ in (char, prev, skc, orient, results):
    df_["SINH_ID"] = as_int64(df_["SINH_ID"])

# ----------------- Presence flags (from source availability) -----------------
has_characteristics = set(char["SINH_ID"].dropna())
has_previous        = set(prev["SINH_ID"].dropna())
has_skc             = set(skc["SINH_ID"].dropna())
has_orientation     = set(orient["SINH_ID"].dropna())
has_results         = set(results["SINH_ID"].dropna())

# ----------------- LEFT merges (no suffixes) -----------------
first_merge = left_merge_no_suffix(first_merge, char,    on="SINH_ID")
first_merge = left_merge_no_suffix(first_merge, prev,    on="SINH_ID")
first_merge = left_merge_no_suffix(first_merge, skc,     on="SINH_ID")
first_merge = left_merge_no_suffix(first_merge, orient,  on="SINH_ID")
first_merge = left_merge_no_suffix(first_merge, results, on="SINH_ID")

# ----------------- Presence flags (added at the end) -----------------
first_merge["has_characteristics"] = first_merge["SINH_ID"].isin(has_characteristics).astype("Int8")
first_merge["has_previous"]        = first_merge["SINH_ID"].isin(has_previous).astype("Int8")
first_merge["has_skc"]             = first_merge["SINH_ID"].isin(has_skc).astype("Int8")
first_merge["has_orientation"]     = first_merge["SINH_ID"].isin(has_orientation).astype("Int8")
first_merge["has_results"]         = first_merge["SINH_ID"].isin(has_results).astype("Int8")

print("Base students:", base_n)
print("After merges :", len(first_merge))  # should match base (one_to_one enforced)


In [ ]:
# first_merge is the new dataframe after this merge,
# later it will be merged with the non-primary key tables using composites
first_merge

In [ ]:
first_merge.columns

In [ ]:
#check if sdo2 merged well
print (first_merge["sdo2_orientation_Number_of_Events_Attended"].sum(min_count=1))
print(first_merge["sdo2_orientation_Number_of_Event_Types"].sum(min_count=1))

#### 7.3.3 Merge other tables
Merge other tables that were not directly able to merge in the first merge (no SINH_ID present). 

In [ ]:
sdo4_dsa.columns

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Identify and evaluate non-SINH_ID columns that 
# can serve as reliable join keys across the datasets by 
# standardizing formats and testing uniqueness, overlap, and coverage.
# first_merge dataframe = sdo1_characteristics + sdo1_previous + sdo2_orientation + sdo2_skc + sdo5_degree + sdo6_results
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# Top 3 matches per dataset pair ONLY, with columns ordered as:
# DF A | DF B | Column_pair | Name_sim | Score | Overlap
# -----------------------------------------------------------------------------

# ---- DataFrames to compare (must already exist as globals) ----
df_names = [
    "sdo1_distance",
    "sdo4_dsa",
    "sdo4_nf",
    "first_merge"
]

# Build registry ONLY from the requested names and only if each exists & is a DataFrame
df_registry = {
    n: globals()[n]
    for n in df_names
    if (n in globals() and isinstance(globals()[n], pd.DataFrame))
}

# Sanity check: ensure we actually have something to compare
if len(df_registry) < 2:
    raise RuntimeError(
        f"Need at least two existing DataFrames from {df_names} to compare. "
        f"Found: {list(df_registry.keys())}"
    )

# ---- Always start fresh: remove stale cached results if present ----
if "similar_pairs" in globals():
    del globals()["similar_pairs"]

# ---- Build similar_pairs for the CURRENT df set only (no cache leakage) ----
NAME_THR, VALUE_THR = 0.60, 0.60
MAX_UNIQUE = 10_000
RNG = np.random.default_rng(42)

def name_similarity(a: str, b: str) -> float:
    toks = lambda s: set(re.findall(r"[a-z0-9]+", str(s).lower()))
    A, B = toks(a), toks(b)
    if not A and not B:
        return 0.0
    j = len(A & B) / len(A | B) if (A | B) else 0.0
    na = re.sub(r"[^a-z0-9]", "", str(a).lower())
    nb = re.sub(r"[^a-z0-9]", "", str(b).lower())
    return 1.0 if na and na == nb else j

def std_uniques(s: pd.Series) -> set:
    # Normalize values to a comparable canonical set with bounded size
    s_num = pd.to_numeric(s, errors="coerce")
    if s_num.notna().mean() > 0.9 or getattr(s.dtype, "kind", "") in "ifub":
        vals = s_num.dropna().unique()
        def fmt(x):
            xf = float(x)
            return str(int(xf)) if xf.is_integer() else str(xf)
        arr = np.array([fmt(v) for v in vals], dtype=object)
    else:
        arr = s.astype(str).str.strip().str.lower()
        arr = arr[arr.ne("nan")].unique()
    if len(arr) > MAX_UNIQUE:
        arr = arr[RNG.choice(len(arr), size=MAX_UNIQUE, replace=False)]
    return set(arr.tolist())

rows = []
for (dfa_name, dfa), (dfb_name, dfb) in combinations(df_registry.items(), 2):
    # Skip empty dataframes defensively
    if dfa is None or dfb is None or dfa.shape[1] == 0 or dfb.shape[1] == 0:
        continue

    for ca in dfa.columns:
        sa = dfa[ca]
        if sa.notna().sum() == 0:
            continue
        Au = std_uniques(sa)
        if not Au:
            continue

        for cb in dfb.columns:
            sb = dfb[cb]
            if sb.notna().sum() == 0:
                continue
            Bu = std_uniques(sb)
            if not Bu:
                continue

            inter = Au & Bu
            if not inter:
                continue
            union = Au | Bu

            containment = len(inter) / min(len(Au), len(Bu))
            jacc = len(inter) / len(union)
            value_score = max(containment, jacc)
            ns = name_similarity(ca, cb)

            if (ns >= NAME_THR and value_score >= VALUE_THR) or value_score >= 0.90:
                rows.append({
                    "df1": dfa_name, "col1": str(ca),
                    "df2": dfb_name, "col2": str(cb),
                    "name_sim": ns,
                    "value_containment": containment,
                    "value_jaccard": jacc,
                    "value_score": value_score,
                    "intersection": len(inter),
                    "unique_df1": len(Au),
                    "unique_df2": len(Bu),
                })

similar_pairs = (
    pd.DataFrame(rows)
    .sort_values(["value_score", "name_sim", "intersection"], ascending=[False, False, False])
    .reset_index(drop=True)
)

# (Extra safety) Filter to the current df set in case anything slipped in
_keep = set(df_registry.keys())
if not similar_pairs.empty:
    similar_pairs = similar_pairs[
        similar_pairs["df1"].isin(_keep) & similar_pairs["df2"].isin(_keep)
    ].reset_index(drop=True)

# ---- Top 3 per dataset pair, with requested column order ----
top_per_dfpair = (
    similar_pairs
    .sort_values(["value_score", "name_sim", "intersection"], ascending=[False, False, False])
    .groupby(["df1", "df2"], as_index=False)
    .head(3)
    .assign(
        Column_pair=lambda d: d["df1"] + "." + d["col1"] + "  ↔  " + d["df2"] + "." + d["col2"],
        Name_sim=lambda d: (d["name_sim"] * 100).round(1).astype(str) + "%",
        Score=lambda d: (d["value_score"] * 100).round(1).astype(str) + "%",
        Overlap=lambda d: d["intersection"].astype(int).astype(str)
                           + " / "
                           + d[["unique_df1", "unique_df2"]].min(axis=1).astype(int).astype(str),
    )
    .loc[:, ["df1", "df2", "Column_pair", "Name_sim", "Score", "Overlap"]]
    .rename(columns={"df1": "DF A", "df2": "DF B"})
)

print("\nTop 3 matches per dataset pair:")
display(top_per_dfpair)


## 8. Merge SDO4 Data (DSA and NF)

### 8.1 Overview
SDO4 contains degree information from two sources:
- **DSA**: Degree, Student, Achievement data
- **NF**: Alternative degree tracking system

Both need to be merged with the main dataset using student and program identifiers.

### 8.2 Prepare DSA Data

In [ ]:
print(first_merge["sdo5_degree_COLLEGEJAAR"].value_counts(dropna=False))

In [ ]:
print(sdo4_dsa["sdo4_dsa_Collegejaar"].value_counts(dropna=False))

### 8.3 Prepare NF Data

### 8.4 Check Data Overlap

In [ ]:
print(first_merge["sdo5_degree_degree_code"].value_counts(dropna=False))

In [ ]:
print(sdo4_dsa["sdo4_dsa_CROHO_code"].value_counts(dropna=False))

### 8.5 Merge DSA Data

### 8.6 Merge NF Data

In [ ]:
print(first_merge["sdo1_previous_Previous_School_Postal_Code"].value_counts(dropna=False))

In [ ]:
print(sdo1_distance["sdo1_distance_postcode"].value_counts(dropna=False))

### 8.7 Combine DSA and NF Data

Since DSA and NF track similar information from different systems, combine them intelligently to create unified degree tracking columns.

In [ ]:
# NaN count per column in sdo1_distance
nan_sums = sdo1_distance.isna().sum().astype(int).sort_values(ascending=False)

# Convert to DataFrame for full display
nan_df = (
    nan_sums.rename("n_missing")
    .reset_index()
    .rename(columns={"index": "column"})
)

# Show all values
nan_df.head(10)


In [ ]:
# NaN count per column in sdo4_dsa
nan_sums = sdo4_dsa.isna().sum().astype(int).sort_values(ascending=False)

# Convert to DataFrame for full display
nan_df = (
    nan_sums.rename("n_missing")
    .reset_index()
    .rename(columns={"index": "column"})
)

# Show all values
nan_df.head(10)


In [ ]:
# NaN count per column in first_merge
nan_sums = first_merge.isna().sum().astype(int).sort_values(ascending=False)

# Convert to DataFrame for full display
nan_df = (
    nan_sums.rename("n_missing")
    .reset_index()
    .rename(columns={"index": "column"})
)

# Show all values
nan_df.head(10)


## 9. Final Data Preparation

### 9.1 Handle Postal Code Conflicts

When merging data from multiple sources, postal codes may conflict. This section resolves those conflicts by:
- Comparing home postal code vs. previous education postal code
- Keeping the most reliable value
- Documenting decisions

In [ ]:
# -----------------------------------------------------------------------------
# Composite-key merges (LEFT, m:1) with normalized keys and SAFE column names:
#   • DSA indicator: [PROGRAM, YEAR] → single Int64 column 'has_dsa'
#   • NF indicator:  [PROGRAM, YEAR] → single Int64 column 'has_nf'
#   • Distances merged TWICE from sdo1_distance, using PC4 (first 4 digits) as Int64:
#       1) previous-school postcode  (first_merge.sdo1_previous_Previous_School_Postal_Code)
#          → columns prefixed 'sdo1_prev_distance' (e.g., sdo1_prev_distance_postcode)
#       2) home/student postcode     (first_merge.sdo5_degree_POSTCODE)
#          → columns prefixed 'sdo1_postal_distance' (e.g., sdo1_postal_distance_latitude)
# Invariants: row count preserved; no suffix collisions; keys normalized.
# -----------------------------------------------------------------------------

import pandas as pd

# ---- helpers ----
def norm_year(s: pd.Series) -> pd.Series:
    # Extract a 4-digit year from formats like "2023/2024"
    return s.astype("string").str.extract(r"(\d{4})", expand=False)

def norm_prog_code(s: pd.Series) -> pd.Series:
    # Keep digits only; zero-pad later so codes align (e.g., 1234 vs 01234)
    return s.astype("string").str.strip().str.extract(r"(\d+)", expand=False)

def norm_postcode_pc4_int(s: pd.Series) -> pd.Series:
    """
    Make postcode a PC4 (first 4 digits) as nullable Int64.
    Non-matching values -> <NA>.
    """
    digits = s.astype("string").str.replace(r"\D+", "", regex=True)
    pc4 = digits.str[:4].where(digits.str.len() >= 4)
    return pc4.astype("Int64")

def assert_no_overlap(left: pd.DataFrame, right: pd.DataFrame, ignore) -> None:
    overlap = (set(left.columns) & set(right.columns)) - set(ignore)
    if overlap:
        raise ValueError(
            "Column name collision without suffixes: "
            f"{sorted(overlap)}. Rename/drop before merging."
        )

# ---- working copies ----
co   = first_merge.copy()
dsa  = sdo4_dsa.copy()
nf   = sdo4_nf.copy()   
dist = sdo1_distance.copy()

# ---- required columns ----
co_year_col    = "sdo5_degree_COLLEGEJAAR"
co_prog_col    = "sdo5_degree_degree_code"
co_prevpc_col  = "sdo1_previous_Previous_School_Postal_Code"  # previous-school postcode
co_homepc_col  = "sdo5_degree_POSTCODE"                        # home/student postcode

dsa_year_col   = "sdo4_dsa_Collegejaar"
dsa_prog_col   = "sdo4_dsa_CROHO_code"

nf_year_col    = "sdo4_nf_Collegejaar"
nf_prog_col    = "sdo4_nf_CROHO"

dist_pc_col    = "sdo1_distance_postcode"

for df_name, df_obj, required in [
    ("first_merge", co,  [co_year_col, co_prog_col, co_prevpc_col, co_homepc_col]),
    ("sdo4_dsa",    dsa, [dsa_year_col, dsa_prog_col]),
    ("sdo4_nf",     nf,  [nf_year_col, nf_prog_col]),
    ("sdo1_distance", dist, [dist_pc_col]),
]:
    missing = [c for c in required if c not in df_obj.columns]
    if missing:
        raise KeyError(f"{df_name} missing required columns: {missing}")

# ---- normalize join keys ----
co["__YEAR__"]      = norm_year(co[co_year_col])
co["__PROG__"]      = norm_prog_code(co[co_prog_col])
co["__PC4_PREV__"]  = norm_postcode_pc4_int(co[co_prevpc_col])   # PC4 Int64
co["__PC4_HOME__"]  = norm_postcode_pc4_int(co[co_homepc_col])   # PC4 Int64

dsa["__YEAR__"] = norm_year(dsa[dsa_year_col])
dsa["__PROG__"] = norm_prog_code(dsa[dsa_prog_col])

nf["__YEAR__"]  = norm_year(nf[nf_year_col])
nf["__PROG__"]  = norm_prog_code(nf[nf_prog_col])

# Build PC4 Int64 key for distance table
dist["__PC4__"] = norm_postcode_pc4_int(dist[dist_pc_col])

# Zero-pad program codes to same width on both sides
width = int(pd.concat(
    [co["__PROG__"].dropna().map(len), dsa["__PROG__"].dropna().map(len), nf["__PROG__"].dropna().map(len)],
    ignore_index=True
).max() or 0)
if width > 0:
    co["__PROG__"]  = co["__PROG__"].str.zfill(width)
    dsa["__PROG__"] = dsa["__PROG__"].str.zfill(width)
    nf["__PROG__"]  = nf["__PROG__"].str.zfill(width)

# ---- deduplicate RHS to their keys (m:1) ----
# DSA → unique key table with indicator
dsa_keys = (
    dsa.dropna(subset=["__PROG__", "__YEAR__"])
       .drop_duplicates(subset=["__PROG__", "__YEAR__"])
       .loc[:, ["__PROG__", "__YEAR__"]]
       .assign(has_dsa=1)
)

nf_keys = (
    nf.dropna(subset=["__PROG__", "__YEAR__"])
      .drop_duplicates(subset=["__PROG__", "__YEAR__"])
      .loc[:, ["__PROG__", "__YEAR__"]]
      .assign(has_nf=1)
)

# Distances → unique per PC4 (Int64)
dist_dim = (
    dist.dropna(subset=["__PC4__"])
        .sort_values("__PC4__")
        .drop_duplicates(subset=["__PC4__"], keep="first")
)

# ---- merge DSA (LEFT, m:1) → single indicator column ----
rows_before = len(co)
assert_no_overlap(co, dsa_keys, ignore={"__PROG__", "__YEAR__"})
co = co.merge(dsa_keys, on=["__PROG__", "__YEAR__"], how="left", validate="m:1")
assert len(co) == rows_before, "Row count changed after DSA merge."
co["has_dsa"] = co["has_dsa"].fillna(0).astype("Int64")

# ---- merge NF (LEFT, m:1) → single indicator column ----
rows_before = len(co)
assert_no_overlap(co, nf_keys, ignore={"__PROG__", "__YEAR__"})
co = co.merge(nf_keys, on=["__PROG__", "__YEAR__"], how="left", validate="m:1")
assert len(co) == rows_before, "Row count changed after NF merge."
co["has_nf"] = co["has_nf"].fillna(0).astype("Int64")

# ---- distance merge #1: previous-school PC4 → rename to sdo1_prev_distance* ----
rows_before = len(co)
dist_cols_to_bring = [c for c in dist.columns if c.startswith("sdo1_distance_")]
assert_no_overlap(co, dist_dim[["__PC4__"] + dist_cols_to_bring], ignore={"__PC4__"})
co = co.merge(
    dist_dim[["__PC4__"] + dist_cols_to_bring],
    left_on="__PC4_PREV__", right_on="__PC4__", how="left", validate="m:1"
)
assert len(co) == rows_before, "Row count changed after distance merge (previous-school)."

rename_prev = {
    c: "sdo1_prev_distance" + c[len("sdo1_distance"):]
    for c in dist_cols_to_bring if c in co.columns
}
co.rename(columns=rename_prev, inplace=True)
co.drop(columns=["__PC4__"], inplace=True, errors="ignore")

# ---- distance merge #2: home/student PC4 → rename to sdo1_postal_distance* ----
rows_before = len(co)
assert_no_overlap(co, dist_dim[["__PC4__"] + dist_cols_to_bring], ignore={"__PC4__"})
co = co.merge(
    dist_dim[["__PC4__"] + dist_cols_to_bring],
    left_on="__PC4_HOME__", right_on="__PC4__", how="left", validate="m:1"
)
assert len(co) == rows_before, "Row count changed after distance merge (home)."

rename_home = {
    c: "sdo1_postal_distance" + c[len("sdo1_distance"):]
    for c in dist_cols_to_bring if c in co.columns
}
co.rename(columns=rename_home, inplace=True)
co.drop(columns=["__PC4__"], inplace=True, errors="ignore")

# ---- cleanup helpers ----
co.drop(columns=["__PROG__", "__YEAR__", "__PC4_PREV__", "__PC4_HOME__"],
        inplace=True, errors="ignore")

# Final result
data = co
data


In [ ]:
# Verify exactly what got added and confirm uniqueness

orig_cols = set(first_merge.columns)
final_cols = set(data.columns)

added = sorted(final_cols - orig_cols)
print(f"Original columns: {len(orig_cols)}")
print(f"Final columns:    {len(final_cols)}")
print(f"Added columns:    {len(added)}")
print("\nNew columns:\n - " + "\n - ".join(added))

# Sanity: ensure no duplicates in final
assert len(data.columns) == len(set(data.columns)), "Duplicate column names found!"


### 9.2 Clean Column Names and Standardize Format

In [ ]:
# Check equality only where both exist
mask = data["sdo1_postal_distance_postcode"].notna() & data["sdo1_prev_distance_postcode"].notna()
equal_rows = (data.loc[mask, "sdo1_postal_distance_postcode"] 
              == data.loc[mask, "sdo1_prev_distance_postcode"]).sum()

total_rows = mask.sum()
print(f"Rows where both postcodes exist: {total_rows}")
print(f"Rows where they are identical:   {equal_rows}")
print(f"→ {(equal_rows / total_rows * 100):.2f}% have the same postcode")


In [ ]:
# inspect whether distances also match when postcodes match

same_dist = (
    data.loc[mask & (data["sdo1_postal_distance_postcode"] == data["sdo1_prev_distance_postcode"]),
             ["sdo1_postal_distance_distance_to_3584_CS",
              "sdo1_prev_distance_distance_to_3584_CS"]]
    .eval("diff = sdo1_postal_distance_distance_to_3584_CS - sdo1_prev_distance_distance_to_3584_CS")
    .assign(equal=lambda x: x["diff"].abs() < 1e-9)
)

print(f"Among identical postcodes: {same_dist['equal'].mean()*100:.2f}% have same distance values")


### 9.3 Remove Redundant/Duplicate Columns

In [ ]:
# -----------------------------------------------------------------------------
# Purpose:
# Verify the correctness of the DSA merge by comparing normalized join keys
# between the main dataset (`data`) and the DSA reference table (`sdo4_dsa`).
#
# Description:
# - Recalculates normalized join keys:
#       • Program code (__PROG__) → digits only, zero-padded.
#       • College year (__YEAR__) → extracted 4-digit year from values like "2021/2022".
# - Counts unique (PROGRAM, YEAR) combinations on each side to understand key coverage.
# - Checks for missing values (NA) in the normalized keys for both datasets.
# - Calculates the coverage rate: the share of (CROHO_code, Collegejaar) pairs from
#   `data` that exist in `sdo4_dsa`.
# - Prints a small random sample of key pairs to show which ones matched successfully.
#
# Outcome:
# Confirms that year and program-code normalization is consistent across both tables,
# verifies that the merge used for `has_dsa` behaves correctly, and ensures the
# DSA indicator accurately reflects true participation based on (PROGRAM, YEAR) keys.
# ----------------------------------------------------------------------------- 



# --- recompute normalized keys (do NOT rely on temp cols in `data`) ---
co_keys = (
    data[[co_year_col, co_prog_col]]
      .assign(__YEAR__ = norm_year(data[co_year_col]),
              __PROG__ = norm_prog_code(data[co_prog_col]))
      [["__PROG__", "__YEAR__"]]
)

dsa_keys = (
    dsa[[dsa_year_col, dsa_prog_col]]
      .assign(__YEAR__ = norm_year(dsa[dsa_year_col]),
              __PROG__ = norm_prog_code(dsa[dsa_prog_col]))
      [["__PROG__", "__YEAR__"]]
)

# 1) How many (PROG, YEAR) pairs on each side (after normalization)?
print("co keys:", co_keys.dropna().drop_duplicates().shape[0])
print("dsa keys:", dsa_keys.dropna().drop_duplicates().shape[0])

# 2) NA rates in normalized keys
print("co __PROG__ NA %:", co_keys["__PROG__"].isna().mean())
print("co __YEAR__ NA %:", co_keys["__YEAR__"].isna().mean())
print("dsa __PROG__ NA %:", dsa_keys["__PROG__"].isna().mean())
print("dsa __YEAR__ NA %:", dsa_keys["__YEAR__"].isna().mean())

# 3) Coverage: what share of co-keys exist in dsa-keys?
coverage = (
    co_keys.dropna().drop_duplicates()
           .merge(dsa_keys.dropna().drop_duplicates(),
                  on=["__PROG__", "__YEAR__"], how="left", indicator=True)
           ["_merge"].eq("both").mean()
)
print(f"Share of co keys found in sdo4_dsa: {coverage:.2%}")

# 4) Probe a few examples
sample = co_keys.dropna().drop_duplicates().sample(n=min(5, len(co_keys)), random_state=42)
probe = sample.merge(dsa_keys.dropna().drop_duplicates(),
                     on=["__PROG__", "__YEAR__"], how="left", indicator=True)
print(probe)


### 9.4 Reorder Columns for Better Readability

Organize columns in a logical order:
1. **Identifiers**: Student ID, degree codes, etc.
2. **Demographics**: Age, gender, nationality, postal code
3. **Academic**: Previous education, course results, dropout status
4. **Engagement**: Orientation attendance, SKC attendance
5. **Derived**: Distance calculations, flags

In [ ]:
print(data["has_dsa"].value_counts(dropna=False))
print((data["has_dsa"]==1).mean())  # overall share of students flagged as DSA


In [ ]:
data.columns

In [ ]:
# check if merged went well
print(data["sdo2_orientation_Number_of_Events_Attended"].sum(min_count=1))
print(data["sdo2_orientation_Number_of_Event_Types"].sum(min_count=1))

In [ ]:
print(data["has_results"].sum(min_count=1))
print(data["has_orientation"].sum(min_count=1))
print(data["has_skc"].sum(min_count=1))
print(data["has_previous"].sum(min_count=1))
print(data["has_characteristics"].sum(min_count=1))

In [ ]:
# NaN count per column in students
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False)

# Convert to DataFrame for full display
nan_df = (
    nan_sums.rename("n_missing")
    .reset_index()
    .rename(columns={"index": "column"})
)

# Show all values
nan_df.head(60)


## 10. Final Dataset Summary and Export

### 10.1 Dataset Overview

In [ ]:
data.SINH_ID.nunique()

### 10.2 Export Combined Dataset

In [ ]:
out_path = OUT_DIR / "v2_combined.csv"
data.to_csv(out_path, index=False)
# upload it to sharepoint and remove data from processed folder.